# Practical 14 — Financial Text Summarization and Information Extraction

A summarize-and-extract pipeline for company filings, run **end to end and fully
offline** on a bundled fictional earnings release (Orion Dynamics Inc., Q3 FY2025).
The deterministic tools in `tools/` do all the work: a regex extractor pulls named
numeric fields and validates them against a small JSON-schema-style contract, then a
grader scores any summary for **faithfulness** (IDF-weighted grounding) and
**figure coverage** (every number must appear verbatim in the source).

In the real project you drive this with an agent (Claude Code / Cline) that chooses
inputs and interprets outputs. This notebook is the Colab-friendly counterpart: it
calls the same reference tools directly so you can watch the whole Extract →
Summarize → Grade → Check loop run and see its real outputs.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "14-financial-text-summarization"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — Load the bundled filing

The source is a fictional Q3 FY2025 earnings release for Orion Dynamics Inc. The
helpers in `tools/_common.py` load every `.txt` in `data/filings/` and concatenate
them. Everything below reads this text only — no network, no API key.

In [ ]:
import json
from tools._common import load_filings, load_filing_text

filings = load_filings()
print("Filings found:", list(filings.keys()))
source = load_filing_text()
print(f"\nSource length: {len(source)} chars\n")
print(source)

## Step 1 — Extract named fields with source spans

`tools.extract.extract` runs one anchored regex per field. Each pattern is tied to
its label (e.g. "Total revenue was ..."), so distractor numbers like the prior-year
margin (54%) or the 22% growth rate are *not* mis-captured. For every match it
records the typed `value`, the `span` into the whitespace-normalised source, and the
matched `source` snippet. A field that matches nothing is simply absent — never guessed.

In [ ]:
from tools.extract import extract

record = extract(source)
print(json.dumps(record, indent=2))

Note the types: `revenue` and `guidance` are strings, while `gross_margin` (58.0) and
`eps` (1.27) are cast to numbers — that typing is what the schema check below relies on.

In [ ]:
print(f"{'field':<14}{'value':<34}span")
print("-" * 60)
for name, entry in record.items():
    print(f"{name:<14}{str(entry['value']):<34}{entry['span']}")

## Step 2 — Validate the extraction against the schema

Before any summary is written, `tools.extract.validate` checks the record against a
small contract: required fields must be present, numbers must fall in range
(a gross margin must be 0–100), and strings must match their pattern. An empty error
list means the extraction is trustworthy. `validate_or_raise` turns any failure into a
`SchemaError` so a bad extraction can stop the pipeline.

In [ ]:
from tools.extract import validate, validate_or_raise, SchemaError

errors = validate(record)
print("schema_errors:", errors)
print("valid:", validate_or_raise(record) is record)

### Try it: corrupt the extraction and re-validate

The README suggests dropping a field or giving `gross_margin` a string / impossible
value. The schema rejects each before a summary is ever written.

In [ ]:
import copy

# (a) impossible margin of 580%
bad = copy.deepcopy(record)
bad["gross_margin"]["value"] = 580.0
print("580% margin ->", validate(bad))

# (b) wrong type: a string where a number is required
bad = copy.deepcopy(record)
bad["gross_margin"]["value"] = "fifty-eight percent"
print("string margin ->", validate(bad))

# (c) drop a required field
bad = copy.deepcopy(record)
bad.pop("eps")
print("missing eps  ->", validate(bad))

try:
    validate_or_raise(bad)
except SchemaError as e:
    print("\nvalidate_or_raise raised SchemaError:", e)

## Step 3 — Write a grounded summary from the fields only

The summarizer's rule: every figure must come from the validated record, and nothing
else. Here we assemble such a summary straight from `record` — the same values, no
outside knowledge, no rounding.

In [ ]:
faithful_summary = (
    f"Orion Dynamics reported revenue of {record['revenue']['value']} (field: revenue), "
    f"a gross margin of {record['gross_margin']['value']:.0f}% (field: gross_margin), and "
    f"diluted EPS of ${record['eps']['value']} (field: eps). "
    f"It raised full-year revenue guidance to a range of {record['guidance']['value']} "
    f"(field: guidance)."
)
print(faithful_summary)

## Step 4 — Grade the summary

`tools.grade.grade` returns two reproducible numbers in [0, 1] plus a verdict:

* **faithfulness** — IDF-weighted share of the summary's tokens that appear in the
  filing. Distinctive tokens (numbers like "58") carry most of the weight, so a
  fabricated figure tanks the score instead of hiding behind common words.
* **figure_coverage** — the blunt literal check: every numeric figure in the summary
  must appear verbatim in the source. `unsupported_figures` lists any that don't.

`supported` is True only when faithfulness ≥ 0.7 **and** figure_coverage == 1.0.

In [ ]:
from tools.grade import grade, faithfulness, figure_coverage, unsupported_figures

result = grade(faithful_summary, source)
print(json.dumps(result, indent=2))

## Step 5 — Check: what happens when the summary invents a figure?

This is the README's first "thing to try": have the summary state a number that is
**not** in the fields (here revenue of $2.30 billion, margin 73%, EPS $4.10, plus
claims absent from the filing). Watch `figure_coverage` fall below 1.0, the invented
numbers surface in `unsupported_figures`, and the verdict flip to unsupported — which
is exactly the signal the agent loop uses to send the summary back for revision.

In [ ]:
invented_summary = (
    "Orion Dynamics posted revenue of $2.30 billion, a gross margin of 73%, and "
    "diluted EPS of $4.10. Management warned of a goodwill writedown in Brazil and "
    "suspended its buyback program amid a regulatory probe in Singapore."
)
invented_result = grade(invented_summary, source)
print(json.dumps(invented_result, indent=2))

Side by side, the difference is stark: the grounded summary clears both gates while
the invented one fails on figure coverage and drops sharply on faithfulness.

In [ ]:
print(f"{'summary':<12}{'faithfulness':<14}{'figure_cov':<12}{'supported'}")
print("-" * 50)
for label, summ in [("faithful", faithful_summary), ("invented", invented_summary)]:
    r = grade(summ, source)
    print(f"{label:<12}{r['faithfulness']:<14}{r['figure_coverage']:<12}{r['supported']}")

print("\nunsupported figures in the invented summary:",
      unsupported_figures(invented_summary, source))

## Recap / next steps

We ran the full Chapter 14 loop offline on the bundled filing:

1. **Load** the Orion Dynamics earnings release (`tools._common.load_filing_text`).
2. **Extract** four named fields with anchored regexes and source spans
   (`tools.extract.extract`).
3. **Validate** them against a JSON-schema-style contract; bad records are rejected
   before any summary is written (`tools.extract.validate` / `validate_or_raise`).
4. **Summarize** using only the validated values, citing each field.
5. **Grade** for faithfulness and figure coverage, and saw an invented-figure summary
   fail the coverage gate (`tools.grade.grade`).

Things to explore next, straight from the README:

* Reword a figure's label in `data/filings/` and re-run — the anchored regex stops
  matching, so the field goes *missing* rather than grabbing the wrong number.
* Add a second `.txt` filing to `data/filings/` and summarise across both
  (`load_filings` already concatenates every file it finds).
* Run the offline test suite to see all of this asserted: `python -m pytest -q`.